In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Locate project root
current_dir = Path.cwd()

if (current_dir / "models").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "models").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate Traffic_Sign_Recognition project root."
    )

MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ROBUSTNESS_DIR = OUTPUTS_DIR / "robustness"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Models directory:", MODELS_DIR)
print("Reports directory:", REPORTS_DIR)

Project root: c:\Projects\Traffic_Sign_Recognition
Models directory: c:\Projects\Traffic_Sign_Recognition\models
Reports directory: c:\Projects\Traffic_Sign_Recognition\reports


In [2]:
# Load consolidated model comparison

comparison_df = pd.read_csv(
    ROBUSTNESS_DIR / "final_model_performance_comparison.csv"
)

display(comparison_df)

,Model,Clean,gaussian_blur,brightness,occlusion,combined,Average Corrupted Accuracy,Overall Average Accuracy
0,Baseline CNN,0.991845,0.992082,0.982977,0.882344,0.854553,0.927989,0.940760
1,EfficientNetB0,0.753919,0.743310,0.742201,0.442359,0.421615,0.587371,0.620681
2,CNN with Heavy Augmentation,0.991211,0.987411,0.978543,0.872922,0.841884,0.920190,0.934394


In [3]:
# Select model based on average corrupted accuracy

best_model = comparison_df.loc[
    comparison_df["Average Corrupted Accuracy"].idxmax()
]

recommended_model = best_model["Model"]

model_file_mapping = {
    "Baseline CNN": "cnn_baseline.keras",
    "EfficientNetB0": "efficientnet_b0.keras",
    "CNN with Heavy Augmentation": "cnn_heavy_augmentation.keras"
}

recommended_model_file = model_file_mapping[recommended_model]

print("RECOMMENDED MODEL")
print("-----------------")
print("Model:", recommended_model)
print("Model file:", recommended_model_file)
print(
    "Clean accuracy:",
    f"{best_model['Clean'] * 100:.2f}%"
)
print(
    "Average corrupted accuracy:",
    f"{best_model['Average Corrupted Accuracy'] * 100:.2f}%"
)
print(
    "Overall average accuracy:",
    f"{best_model['Overall Average Accuracy'] * 100:.2f}%"
)

RECOMMENDED MODEL
-----------------
Model: Baseline CNN
Model file: cnn_baseline.keras
Clean accuracy: 99.18%
Average corrupted accuracy: 92.80%
Overall average accuracy: 94.08%


In [4]:
recommendation_text = f"""
# Traffic Sign Recognition — Deployment Recommendation

## 1. Recommended Model

**Model:** {recommended_model}

**Model file:** `models/{recommended_model_file}`

The Baseline CNN is recommended for deployment because it achieved
the highest average accuracy across the four corrupted test conditions.

## 2. Performance Summary

| Metric | Result |
|---|---:|
| Clean test accuracy | {best_model['Clean'] * 100:.2f}% |
| Average corrupted accuracy | {best_model['Average Corrupted Accuracy'] * 100:.2f}% |
| Overall average accuracy | {best_model['Overall Average Accuracy'] * 100:.2f}% |

## 3. Why This Model Was Selected

The Baseline CNN achieved:
- Strong clean-test accuracy of 99.18%.
- The highest average corrupted-test accuracy among the three models.
- Better overall average accuracy than the EfficientNetB0 and
  heavy-augmentation CNN models in this evaluation.

## 4. Robustness Considerations

The Baseline CNN performed well under Gaussian blur and brightness
changes. However, its accuracy decreased under occlusion and combined
corruption.

This indicates that the model can be sensitive when important parts
of a traffic sign are blocked or when multiple image distortions occur.

## 5. Deployment Recommendation

Use `cnn_baseline.keras` as the primary model for a prototype
traffic-sign recognition application.

Before deployment in a real vehicle or safety-critical environment,
additional testing is necessary, including:
- Testing on traffic signs captured in real-world driving conditions.
- Evaluating different lighting, weather, distance, and camera angles.
- Testing partially hidden, damaged, and unfamiliar signs.
- Measuring prediction latency and memory requirements on the target device.
- Adding a safe response for low-confidence or uncertain predictions.

The model should be treated as a prototype, not as a validated
safety-critical driving system.

## 6. Final Conclusion

The Baseline CNN provides the best balance of clean-test accuracy
and average corrupted-test accuracy among the three evaluated models.
It is therefore the recommended model for the next deployment
prototype stage.
"""

report_path = REPORTS_DIR / "deployment_recommendation.md"

with open(report_path, "w", encoding="utf-8") as file:
    file.write(recommendation_text)

print("Deployment recommendation report saved successfully.")
print("File:", report_path)

Deployment recommendation report saved successfully.
File: c:\Projects\Traffic_Sign_Recognition\reports\deployment_recommendation.md
